## Refactoring Checklist
 
### Issues Found:
- [ ] Function `X` does too much (loads file AND validates AND calls API)
- [ ] Error handling missing in function `Y`
- [ ] Code repeated in functions `A` and `B` (could be helper function)
- [ ] Hardcoded prompt in function `Z`
- [ ] No error message when file not found
- [ ] Validation errors caught but not shown
 
### Priority:
1. [Most critical issue]
2. [Second priority]
3. [Third priority]

---

## Refactoring Checklist
 
### Issues Found:
- [ ] Function process_products does too much: selects products, creates API client, calls API/mock generator, handles errors, prints progress, saves JSON, and summarizes results.
- [ ] Function prepare_huggingface_dataset does too much: loads external dataset, creates folders, extracts/normalizes product data, saves images, generates random prices, writes JSON, and prints status.
- [ ] Function validate_setup mixes validation, file loading, image encoding, environment checking, and console output.
- [ ] Error handling is silent when loading .env: the except Exception: pass hides dotenv/config problems.
- [ ] Validation errors are hidden in normalize_product: invalid prices are silently replaced with 29.99.
- [ ] Code repeated in normalize_product and prepare_huggingface_dataset: both contain product field fallback logic for name, category, id, and metadata.
- [ ] Hardcoded values appear throughout the script: 29.99, 19.99, 149.99, 3, 2.0, 1.0, "products.json", "generated_listings.json", and "ashraq/fashion-product-images-small".
- [ ] Hardcoded prompt content in create_product_listing_prompt and call_openai_vision_api
- [ ] Console output is mixed into business logic using many print() calls, especially in process_products, prepare_huggingface_dataset, and validate_setup.
- [ ] API logic and image/file preparation are mixed in call_openai_vision_api, because it encodes the image, creates the prompt, calls the API, and parses the result.

### Priority:
1. Refactor process_products because it is the main monolithic function and controls the full workflow.
2. Fix silent failures in .env loading and invalid price handling so errors are visible.
3. Move hardcoded values and prompt text into constants or configuration.

import json
from pydantic import BaseModel, ValidationError
from typing import Optional, Dict, Any

# 1. Pydantic Model (Defined outside the functions)
class ProductInput(BaseModel):
    id: str | int
    name: str
    price: float
    category: str
    image_path: str
    additional_info: Optional[str] = None

# 2. File Operations (Separated!)
def load_json_file(file_path: str) -> dict:
    """Loads and parses a JSON file."""
    # Try/except block for FileNotFoundError goes here
    pass

def save_json(data: list | dict, file_path: str) -> None:
    """Saves generated listings or product data into a JSON file."""
    # Try/except block for IOError goes here
    pass

# 3. Validation
def normalize_and_validate_product(product_dict: dict) -> Optional[ProductInput]:
    """Validates product data using the Pydantic model."""
    try:
        return ProductInput(**product_dict)
    except ValidationError as e:
        # Print error details here
        return None

# 4. Prompting
def create_product_listing_prompt(product: ProductInput) -> str:
    """Creates the user prompt for OpenAI using validated product data."""
    return f"Create a listing for {product.name} in the {product.category} category..."

# 5. Parsing API Response
def parse_openai_listing_response(response: Any) -> dict:
    """Takes the OpenAI response and extracts/converts it to a dictionary."""
    # Assuming you are using structured outputs here
    # return response.output_parsed.model_dump()
    pass

# 6. Formatting Output (Success vs Failure)
def build_success_result(product: ProductInput, listing_data: dict, processing_time: float) -> dict:
    """Formats a successful product result."""
    return {
        "status": "success",
        "product_id": product.id,
        "listing_data": listing_data,
        "processing_time": processing_time
    }

def build_failure_result(product_dict: dict, error_type: str, error_message: str) -> dict:
    """Formats a failed product result."""
    return {
        "status": "failure",
        "product_id": product_dict.get("id", "Unknown"),
        "error_type": error_type,
        "error_message": error_message
    }

### TESTING

In [3]:
import json
from pydantic import BaseModel, ValidationError
from typing import Optional, Dict, Any

# ==========================================
# 1. PYDANTIC MODEL
# ==========================================
class ProductInput(BaseModel):
    id: str | int
    name: str
    price: float
    category: str
    image_path: str
    additional_info: Optional[str] = None

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def normalize_and_validate_product(product_dict: dict) -> Optional[ProductInput]:
    """Validates product data using the Pydantic model."""
    try:
        return ProductInput(**product_dict)
    except ValidationError as e:
        # We are adding a nice print statement here so you can see exactly WHAT failed
        print(f"❌ Validation failed for: {product_dict.get('name', 'Unknown Product')}")
        for error in e.errors():
            print(f"   - {error['loc'][0]}: {error['msg']}")
        return None

def create_product_listing_prompt(product: ProductInput) -> str:
    """Creates the user prompt for OpenAI using validated product data."""
    return f"Create a listing for {product.name} in the {product.category} category. It costs ${product.price}."

# ==========================================
# TEST PLAYGROUND
# ==========================================
if __name__ == "__main__":
    
    print("--- TEST 1: Intentional Failure (Missing Data) ---")
    bad_test_data = {
        "name": "Test Product", 
        "price": 99.99
        # Missing id, category, and image_path!
    }
    
    bad_result = normalize_and_validate_product(bad_test_data)
    print(f"Returned object: {bad_result}\n")
    
    
    print("--- TEST 2: Successful Validation ---")
    good_test_data = {
        "id": "PROD-001",
        "name": "Test Product",
        "price": 99.99,
        "category": "Electronics",
        "image_path": "/assets/headphones.png"
    }
    
    good_result = normalize_and_validate_product(good_test_data)
    print(f"Returned object: {good_result}\n")
    
    # Let's test the prompt builder too while we are at it!
    if good_result:
        print("--- TEST 3: Prompt Generation ---")
        prompt = create_product_listing_prompt(good_result)
        print(prompt)

--- TEST 1: Intentional Failure (Missing Data) ---
❌ Validation failed for: Test Product
   - id: Field required
   - category: Field required
   - image_path: Field required
Returned object: None

--- TEST 2: Successful Validation ---
Returned object: id='PROD-001' name='Test Product' price=99.99 category='Electronics' image_path='/assets/headphones.png' additional_info=None

--- TEST 3: Prompt Generation ---
Create a listing for Test Product in the Electronics category. It costs $99.99.


In [4]:
import json
import os

# ==========================================
# 3. FILE HELPER FUNCTIONS
# ==========================================

def load_json_file(file_path: str) -> dict:
    """Loads a JSON file with strict error handling so it NEVER fails silently."""
    try:
        with open(file_path, 'r') as f:
            return json.load(f)
            
    except FileNotFoundError:
        print(f"❌ ERROR in load_json_file(): FileNotFoundError")
        print(f"   Location: '{file_path}' could not be found.")
        print(f"   Suggestion: Check your spelling and make sure the file is in this folder.")
        return {} # Return empty dict so the program doesn't crash
        
    except json.JSONDecodeError as e:
        print(f"❌ ERROR in load_json_file(): JSONDecodeError")
        print(f"   Location: Line {e.lineno}, Column {e.colno} in '{file_path}'")
        print(f"   Suggestion: Your JSON has a syntax error (maybe a missing comma or quote).")
        return {}

def save_json(data: list | dict, file_path: str) -> bool:
    """Saves data to a JSON file and confirms success."""
    try:
        with open(file_path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"✅ SUCCESS: Data saved beautifully to '{file_path}'")
        return True
    except IOError as e:
        print(f"❌ ERROR in save_json(): IOError")
        print(f"   Message: {str(e)}")
        print(f"   Suggestion: Check if you have permission to write to this folder.")
        return False


# ==========================================
# TEST PLAYGROUND: FILE OPERATIONS
# ==========================================
if __name__ == "__main__":
    
    print("--- TEST 4: Intentional Failure (Missing File) ---")
    # We are asking it to load a file that doesn't exist
    bad_data = load_json_file("ghost_products.json")
    print(f"Result returned: {bad_data}\n")
    
    
    print("--- TEST 5: Successful Save & Load ---")
    # 1. Let's create some dummy data to save
    dummy_data = {
        "products": [
            {"id": "PROD-001", "name": "Test Product"}
        ]
    }
    
    # 2. Test saving it
    save_successful = save_json(dummy_data, "test_output.json")
    
    # 3. Test loading the file we just created!
    if save_successful:
        loaded_data = load_json_file("test_output.json")
        print(f"Result loaded from file: {loaded_data}")

--- TEST 4: Intentional Failure (Missing File) ---
❌ ERROR in load_json_file(): FileNotFoundError
   Location: 'ghost_products.json' could not be found.
   Suggestion: Check your spelling and make sure the file is in this folder.
Result returned: {}

--- TEST 5: Successful Save & Load ---
✅ SUCCESS: Data saved beautifully to 'test_output.json'
Result loaded from file: {'products': [{'id': 'PROD-001', 'name': 'Test Product'}]}


### Modularize Functions

## Functions That Do Multiple Things

- [ ] `process_products()` does multiple things:
      - Selects products using `limit`
      - Creates the OpenAI client
      - Loops through each product
      - Calls the OpenAI API or mock generator
      - Handles errors
      - Formats success/failure results
      - Saves JSON after every product
      - Prints progress and summary

- [ ] `prepare_huggingface_dataset()` does multiple things:
      - Imports and loads the HuggingFace dataset
      - Creates the image output directory
      - Extracts product metadata
      - Generates random product prices
      - Saves product images to disk
      - Builds product dictionaries
      - Saves the final products JSON file
      - Prints status messages

- [ ] `validate_setup()` does multiple things:
      - Loads product data
      - Validates that products can be loaded
      - Reads and encodes the first product image
      - Checks whether `OPENAI_API_KEY` exists
      - Prints validation messages

- [ ] `call_openai_vision_api()` does multiple things:
      - Encodes the product image
      - Creates the OpenAI prompt
      - Builds the API request
      - Calls the OpenAI API
      - Parses the structured response

- [ ] `load_products()` does more than only loading:
      - Checks whether the product file exists
      - Reads JSON or CSV files
      - Parses the file contents
      - Normalizes every product record
      - Raises validation errors for unsupported formats

1. `process_products()` - validate/call API/save results/output formatting.
2. `prepare_huggingface_dataset()` - file operations and business logic.
3. `validate_setup()` - load data, validate image, check API key, and print results.

In [ ]:
# BEFORE: One function does everything
def process_products(json_file):
    # Loads JSON
    # Validates data
    # Calls API
    # Processes response
    # Saves results
    pass
 
# AFTER: Separate functions for each concern
def load_and_validate_products(json_path: str) -> list:
    """Load JSON and validate products."""
    # Only handles loading and validation
    pass
 
def generate_description(product: dict, api_client) -> dict:
    """Generate description for one product."""
    # Only handles API call
    pass
 
def process_products(products: list, api_client) -> list:
    """Process all products."""
    # Only orchestrates the processing
    pass
 
def save_results(results: list, output_path: str) -> None:
    """Save results to file."""
    # Only handles file output
    pass

In [29]:
from product_listing_generator_refactored import (
    validate_products_path,
    load_product_rows,
    process_product_rows,
    validate_products,
    load_validate_and_process_products,
    validate_image_file,
    create_product_listing_prompt,
    build_openai_input,
    call_openai_vision_api,
    parse_openai_listing_response,
    generate_product_listing,
    process_single_product,
    process_product_batch,
    build_success_result,
    build_failure_result,
    save_results,
)

modularized_functions = {
    "Loading functions": [
        "validate_products_path()",
        "load_product_rows()",
    ],
    "Validation functions": [
        "validate_products()",
        "validate_image_file()",
    ],
    "Processing functions": [
        "process_product_rows()",
        "process_single_product()",
        "process_product_batch()",
        "load_validate_and_process_products()",
    ],
    "API functions": [
        "create_product_listing_prompt()",
        "build_openai_input()",
        "call_openai_vision_api()",
        "parse_openai_listing_response()",
        "generate_product_listing()",
    ],
    "Output formatting functions": [
        "build_success_result()",
        "build_failure_result()",
    ],
    "Saving functions": [
        "save_results()",
    ],
}

for category, functions in modularized_functions.items():
    print(f"\n{category}")
    for function_name in functions:
        print(f"  - {function_name}")


Loading functions
  - validate_products_path()
  - load_product_rows()

Validation functions
  - validate_products()
  - validate_image_file()

Processing functions
  - process_product_rows()
  - process_single_product()
  - process_product_batch()
  - load_validate_and_process_products()

API functions
  - create_product_listing_prompt()
  - build_openai_input()
  - call_openai_vision_api()
  - parse_openai_listing_response()
  - generate_product_listing()

Output formatting functions
  - build_success_result()
  - build_failure_result()

Saving functions
  - save_results()


In [30]:
# Modular workflow example

products_path = "products.json"

# 1. Validate file path
path = validate_products_path(products_path)

# 2. Load raw data
raw_rows = load_product_rows(path)

# 3. Process/normalize data
processed_products = process_product_rows(raw_rows)

# 4. Validate product data
validated_products = validate_products(processed_products)

# 5. Process products in mock mode
results = process_product_batch(
    products=validated_products[:1],
    client=None,
    model="gpt-4o-mini",
    delay_seconds=0,
    mock=True
)

# 6. Save results
save_results(results, "modular_test_output.json")

print("Modular workflow completed successfully.")
print(results[0])

[1/1] Turtle Check Men Navy Blue Shirt
  ✓ Listing generated
Modular workflow completed successfully.
{'status': 'success', 'product': {'id': 15970, 'name': 'Turtle Check Men Navy Blue Shirt', 'price': 112.3, 'category': 'Shirts', 'image_path': 'product_images/product_15970.jpg', 'additional_info': 'additional_info: gender: Men; baseColour: Navy Blue; season: Fall; usage: Casual'}, 'listing': {'title': 'Turtle Check Men Navy Blue Shirt | Stylish Shirts', 'description': 'Bring a polished upgrade to your everyday essentials with Turtle Check Men Navy Blue Shirt. Designed for shoppers who want style, practicality, and a clean product presentation, this shirts item is positioned as a versatile choice for modern buyers. The listing generator would normally analyze the product image to describe visible details such as color, silhouette, material cues, and design elements. This mock entry confirms that the batch workflow, JSON formatting, and save process are working before using real API cal

## Remodularization Summary

The refactored code separates the original large workflow into smaller helper functions.

- Loading functions only load data.
- Validation functions only validate files, images, and product fields.
- API functions only prepare prompts, build OpenAI inputs, call the API, or parse responses.
- Processing functions only transform or process product data.
- Output formatting functions only build success or failure dictionaries.
- Saving functions only save results.

This makes the code easier to test, debug, and maintain.

### Error Handling

In [5]:
def load_json_file(file_path: str) -> dict:
    """Load and parse JSON file with error handling."""
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
        return data
    except FileNotFoundError:
        error_msg = (
            f"ERROR in load_json_file(): FileNotFoundError\n"
            f"  Location: File '{file_path}' not found\n"
            f"  Suggestion: Check that the file path is correct and the file exists"
        )
        print(error_msg)
        raise
    except json.JSONDecodeError as e:
        error_msg = (
            f"ERROR in load_json_file(): JSONDecodeError\n"
            f"  Location: File '{file_path}', line {e.lineno}, column {e.colno}\n"
            f"  Message: {e.msg}\n"
            f"  Suggestion: Check JSON syntax at the indicated location"
        )
        print(error_msg)
        raise

In [6]:
# Test error handling
try:
    load_json_file("nonexistent.json")  # Should show clear error
except FileNotFoundError:
    pass  # Error message already printed

ERROR in load_json_file(): FileNotFoundError
  Location: File 'nonexistent.json' not found
  Suggestion: Check that the file path is correct and the file exists


## ERROR MESSAGE FORMAT MODIFIED

In [22]:
def load_json_file(file_path: str) -> dict:
    """Load and parse JSON file with error handling."""
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
        return data
    except FileNotFoundError:
        error_msg = (
        f"ERROR in load_json_file(): FileNotFoundError\n"
            f"  Location: File '{file_path}' not found\n"
            f"  Suggestion: Check that the file path is correct and the file exists"
        )
        print(error_msg)
        raise
    except json.JSONDecodeError as e:
        error_msg = (
            f"ERROR in {function_name}(): {error_type}\n"
            f"  Location: {context}\n"
            f"  Message: {error_message}\n"
            f"  Suggestion: {helpful_tip}"
        )
        print(error_msg)
        raise

In [23]:
# Test error handling
try:
    load_json_file("nonexistent.json")  # Should show clear error
except FileNotFoundError:
    pass  # Error message already printed

ERROR in load_json_file(): FileNotFoundError
  Location: File 'nonexistent.json' not found
  Suggestion: Check that the file path is correct and the file exists


In [24]:
import json
from pathlib import Path

from product_listing_generator_refactored import (
    call_openai_vision_api,
    load_validate_and_process_products,
    process_products,
)

# Create test files with the hardcoded names from the template
valid_data = [
    {
        "id": 1,
        "name": "Test Product",
        "price": 29.99,
        "category": "Test Category",
        "image_path": "product_images/product_26960.jpg",
        "additional_info": "Test product for refactoring validation"
    }
]

Path("valid.json").write_text(json.dumps(valid_data, indent=2), encoding="utf-8")

Path("invalid.json").write_text(
    '[{"name": "Broken Product", }]',
    encoding="utf-8"
)

invalid_data = [
    {
        "id": 1,
        "name": "",
        "price": -10,
        "category": "",
        "image_path": ""
    }
]

Path("invalid_data.json").write_text(
    json.dumps(invalid_data, indent=2),
    encoding="utf-8"
)

# Test scenarios
test_cases = [
    ("valid.json", "Should work"),
    ("missing.json", "Should show file not found error"),
    ("invalid.json", "Should show JSON parse error"),
    ("invalid_data.json", "Should show validation errors"),
]

for file_path, expectation in test_cases:
    print(f"\nTEST: {file_path}")
    print(f"Expected: {expectation}")

    try:
        products = load_validate_and_process_products(file_path)
        print(f"PASS: Loaded {len(products)} product(s)")

        if file_path == "valid.json":
            results = process_products(
                products=products,
                output_path="test_generated_listings.json",
                model="gpt-4o-mini",
                limit=1,
                mock=True
            )
            print(f"PASS: Processing result status = {results[0]['status']}")

    except Exception as error:
        print(f"EXPECTED ERROR CAUGHT: {error.__class__.__name__}")


TEST: valid.json
Expected: Should work
PASS: Loaded 1 product(s)

Processing 1 product(s)...
Mode: MOCK / offline
Model: N/A

[1/1] Test Product
  ✓ Listing generated

Done.
Successful listings: 1
Failed listings: 0
Saved to: test_generated_listings.json
PASS: Processing result status = success

TEST: missing.json
Expected: Should show file not found error
ERROR in validate_products_path(): FileNotFoundError
  Location: Product file 'missing.json'
  Message: Product file was not found.
  Suggestion: Check that the file path is correct and the file exists.
EXPECTED ERROR CAUGHT: FileNotFoundError

TEST: invalid.json
Expected: Should show JSON parse error
ERROR in load_product_rows(): JSONDecodeError
  Location: File 'invalid.json', line 1, column 27
  Message: Illegal trailing comma before end of object
  Suggestion: Check JSON syntax at the indicated line and column.
EXPECTED ERROR CAUGHT: JSONDecodeError

TEST: invalid_data.json
Expected: Should show validation errors
ERROR in valida

In [28]:
class FakeResponses:
    def parse(self, **kwargs):
        raise RuntimeError("Simulated API failure for testing")


class FakeClient:
    responses = FakeResponses()


try:
    call_openai_vision_api(
        client=FakeClient(),
        prompt="Hardcoded test prompt",
        image_data_url="data:image/png;base64,abc"
    )
except RuntimeError as error:
    print(f"EXPECTED API ERROR CAUGHT: {error}")

EXPECTED API ERROR CAUGHT: Simulated API failure for testing


## Step 6 Results

- `valid.json` works correctly and produces a mock product listing.
- `missing.json` shows a file not found error.
- `invalid.json` shows a JSON parsing error with line and column.
- `invalid_data.json` shows validation errors for invalid product fields.
- The fake API client confirms API errors can be caught and displayed.

**Checkpoint:** All test cases pass with appropriate error messages.